# This is the fine-tuning notebook

In [1]:
import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = "/kaggle/input/datasets/langchenneto/soco-baseline-pipeline"          
sys.path.insert(0, REPO)

from pathlib import Path
from soco.config import Config
from soco.evaluate import evaluate, sweep_matching, sweep_features
from soco.decide import recommend
from PIL import Image
from torch.utils.data import Dataset

# The competition dataset

In [2]:
class AMLQueryDataset(Dataset):
    """One source query and a set of accepted targets per row."""

    def __init__(self, df, root, max_queries=0):
        self.df = df.reset_index(drop=True).copy()
        if max_queries:
            self.df = self.df.iloc[:max_queries].copy()
        self.root = Path(root)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        src_path = self.root / row["source_image"]
        trg_path = self.root / row["target_image"]

        with Image.open(src_path) as im:
            src_img = im.convert("RGB")
        with Image.open(trg_path) as im:
            trg_img = im.convert("RGB")

        targets = np.asarray(
            json.loads(row["target_points"]), dtype=np.float32
        ).reshape(-1, 2)
        scale = float(row["scale"])

        if len(targets) == 0 or not np.isfinite(targets).all():
            raise ValueError(f"Invalid target_points for {row['id']}")
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"Invalid scale for {row['id']}")

        return {
            "pair_id": str(row["id"]),       # Unique query ID
            "image_pair_id": str(row["pair_id"]),
            "category": "all",
            "src_path": str(src_path),
            "trg_path": str(trg_path),
            "src_img": src_img,
            "trg_img": trg_img,
            "src_kps": np.asarray(
                [[row["source_x"], row["source_y"]]], dtype=np.float32
            ),
            "trg_kps": None,                 # No one-to-one target label
            "trg_bbox": None,
            "accepted_targets": targets,
            "scale": scale,
        }

In [3]:
# Configuration cell:

root = Path("/kaggle/input/competitions/aml-challenge-2026-27")

cfg = Config(
    data_root = str(root),   
    data_format ="csv",
    csv_path = str(root / "test.csv"),
    image_dir = str(root),
    backbone = "dinov2_vitb14",
    # Offline (internet disabled): point to the attached DINOv2 source + weights
    hub_repo ="/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2", hub_local=True,
    weights_path ="/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2_assets/dinov2_vitb14.pth",
    img_size =518, max_pairs=500, out_dir="/kaggle/working/outputs",
)

# The competition metric

In [4]:
import soco.evaluate as eval_module
from soco.metrics import PCKMeter

class AMLPCKMeter(PCKMeter):
    def update(self, sample, pred, flipped=False):
        pred = np.asarray(pred, dtype=np.float64)

        if pred.shape != (1, 2) or not np.isfinite(pred).all():
            raise ValueError(
                f"Expected one finite (x, y) prediction; got {pred}"
            )

        targets = np.asarray(
            sample["accepted_targets"], dtype=np.float64
        )
        distance = np.linalg.norm(targets - pred[0], axis=1).min()
        normalized_error = distance / sample["scale"]

        self.rows.append({
            "pair_id": sample["pair_id"],
            "category": sample["category"],
            "kp": 0,
            "dist": float(normalized_error),
            "flipped": flipped,
        })

    def frame(self):
        df = pd.DataFrame(
            self.rows,
            columns=["pair_id", "category", "kp", "dist", "flipped"],
        )
        for alpha in self.alphas:
            df[f"pck@{alpha}"] = (df["dist"] < alpha).astype(float)
        return df


# evaluate(), sweep_matching(), and sweep_features() use this module global.
eval_module.PCKMeter = AMLPCKMeter

# Held out validation dataset

In [5]:
train = pd.read_csv(root / "train.csv")

pair_ids = train["pair_id"].drop_duplicates().to_numpy()
assert len(pair_ids) >= 2

rng = np.random.default_rng(42)
rng.shuffle(pair_ids)

n_val = min(len(pair_ids) - 1, max(1, int(0.2 * len(pair_ids))))
val_ids = set(pair_ids[:n_val])

is_val = train["pair_id"].isin(val_ids)
train_df = train.loc[~is_val].copy()
val_df = train.loc[is_val].sample(frac=1, random_state=42)

assert set(train_df["pair_id"]).isdisjoint(val_df["pair_id"])

val_ds = AMLQueryDataset(val_df, root, max_queries=500)

sample = val_ds[0]
print("Validation queries:", len(val_ds))
print("Source point:", sample["src_kps"])
print("Accepted targets:", sample["accepted_targets"])
print("Scale:", sample["scale"])

Validation queries: 453
Source point: [[120.      244.91667]]
Accepted targets: [[1395.8823  692.    ]]
Scale: 562.8041286079647


# Stage 1A

In [6]:
assert len(val_ds) > 0
assert eval_module.PCKMeter is AMLPCKMeter

sweep = eval_module.sweep_matching(
    cfg.replace(run_name="s1"),
    {
        "matcher": ["argmax", "soft_window"],
        "feat_upsample": [1, 2],
        "soft_temperature": [0.02, 0.04],
    },
    ds=val_ds,
)

best = sweep.iloc[0][
    ["matcher", "feat_upsample", "soft_temperature"]
].to_dict()

best["feat_upsample"] = int(best["feat_upsample"])

print("Best settings:", best)
print("Validation PCK:", sweep.iloc[0]["pck@0.1 (per-kp)"] / 100)

/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/input/datasets/langchenneto/dinov2-weights-and-repo/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


sweep:   0%|          | 0/453 [00:00<?, ?it/s]

    matcher  feat_upsample  soft_temperature  n_pairs  n_kps  pck@0.05 (per-kp)  pck@0.05 (per-img)  pck@0.1 (per-kp)  pck@0.1 (per-img)  pck@0.15 (per-kp)  pck@0.15 (per-img)  flip_rate
soft_window              2              0.04      453    453              19.65               19.65             38.41              38.41              53.42               53.42        0.0
soft_window              2              0.02      453    453              19.65               19.65             37.75              37.75              53.64               53.64        0.0
     argmax              2              0.04      453    453              17.66               17.66             37.53              37.53              51.88               51.88        0.0
     argmax              2              0.02      453    453              17.66               17.66             37.53              37.53              51.88               51.88        0.0
     argmax              1              0.02      453    453     

# Stage 1B

In [7]:
feat = eval_module.sweep_features(
    cfg.replace(run_name="s1f", **best),
    {
        "layer": [-1, -2],
        "img_size": [448, 518, 644],
    },
    ds=val_ds,
)

display(feat.head())

{'layer': -1, 'img_size': 448} PCK@0.1=35.98
{'layer': -1, 'img_size': 518} PCK@0.1=38.41
{'layer': -1, 'img_size': 644} PCK@0.1=40.40
{'layer': -2, 'img_size': 448} PCK@0.1=26.27
{'layer': -2, 'img_size': 518} PCK@0.1=28.04
{'layer': -2, 'img_size': 644} PCK@0.1=30.68


,layer,img_size,n_pairs,n_kps,pck@0.05 (per-kp),pck@0.05 (per-img),pck@0.1 (per-kp),pck@0.1 (per-img),pck@0.15 (per-kp),pck@0.15 (per-img),flip_rate,sec_per_pair
2,-1,644,453,453,24.282561,24.282561,40.397351,40.397351,52.538631,52.538631,0.0,0.066492
1,-1,518,453,453,19.646799,19.646799,38.410596,38.410596,53.421634,53.421634,0.0,0.036775
0,-1,448,453,453,20.529801,20.529801,35.982340,35.982340,47.461369,47.461369,0.0,0.033225
5,-2,644,453,453,15.231788,15.231788,30.684327,30.684327,36.644592,36.644592,0.0,0.078602
4,-2,518,453,453,14.569536,14.569536,28.035320,28.035320,35.761589,35.761589,0.0,0.042154


# Stage 1C

In [8]:
top = feat.iloc[0]

cfg1 = cfg.replace(
    run_name="stage1_best",
    layer=int(top["layer"]),
    img_size=int(top["img_size"]),
    **best,
)

summ, meter = eval_module.evaluate(cfg1, ds=val_ds)

print("Validation PCK:", summ["pck@0.1 (per-kp)"] / 100)
print(recommend(f"{cfg1.out_dir}/stage1_best", target=70.0))

stage1_best:   0%|          | 0/453 [00:00<?, ?it/s]

{
  "n_pairs": 453,
  "n_kps": 453,
  "pck@0.05 (per-kp)": 24.28,
  "pck@0.05 (per-img)": 24.28,
  "pck@0.1 (per-kp)": 40.4,
  "pck@0.1 (per-img)": 40.4,
  "pck@0.15 (per-kp)": 52.54,
  "pck@0.15 (per-img)": 52.54,
  "flip_rate": 0.0,
  "sec_per_pair": 0.07
}
          pck@0.05  pck@0.1  pck@0.15
category                             
all           24.3     40.4      52.5
Validation PCK: 0.40397350993377484
Run: stage1_best   PCK@0.1 (per-kp) = 40.4   target = 70.0
Weakest categories: all 40.4
DECISION: STAGE 2 - fine-tune the last DINO blocks on the training pairs.
  First make sure the matching sweep (matcher / flip_test / layer / img_size) is done;
  carry its best settings into fine-tuning.
  `python run.py finetune --set ft_unfreeze_blocks=2 ft_epochs=2`


# Stage 2: Fine tuning

In [9]:
from soco.finetune import finetune

cfg2 = cfg1.replace(
    run_name="stage2_ft",
    ft_epochs=1,
    ft_pair_flip_aug=False,
)

train_ds = AMLQueryDataset(train_df, root)

best_val = finetune(
    cfg2,
    train_ds=train_ds,
    val_ds=val_ds,
)

trainable params: 14.18M


epoch 0:   0%|          | 0/510 [00:00<?, ?it/s]

epoch 0: val PCK@0.1 = 58.50


In [10]:
from pathlib import Path

checkpoint = Path(cfg2.out_dir) / cfg2.run_name / "best.pth"
assert checkpoint.is_file(), f"Checkpoint missing: {checkpoint}"
assert eval_module.PCKMeter is AMLPCKMeter

ft_cfg = cfg2.replace(
    ft_checkpoint=str(checkpoint),
    run_name="stage2_ft_reload_check",
)

# No model argument: evaluate loads a fresh model from ft_cfg.
ft_summary, ft_meter = eval_module.evaluate(
    ft_cfg,
    ds=val_ds,
)

assert int(ft_summary["n_kps"]) == len(val_ds)

print("Validation queries:", len(val_ds))
print("Reloaded PCK@0.1:", ft_summary["pck@0.1 (per-kp)"])

stage2_ft_reload_check:   0%|          | 0/453 [00:00<?, ?it/s]

{
  "n_pairs": 453,
  "n_kps": 453,
  "pck@0.05 (per-kp)": 41.28,
  "pck@0.05 (per-img)": 41.28,
  "pck@0.1 (per-kp)": 58.5,
  "pck@0.1 (per-img)": 58.5,
  "pck@0.15 (per-kp)": 66.89,
  "pck@0.15 (per-img)": 66.89,
  "flip_rate": 0.0,
  "sec_per_pair": 0.07
}
          pck@0.05  pck@0.1  pck@0.15
category                             
all           41.3     58.5      66.9
Validation queries: 453
Reloaded PCK@0.1: 58.498896247240616


# Submission

In [11]:
raw = pd.read_csv(root / "test.csv")

adapted = pd.DataFrame({
    "pair_id": raw["id"],
    "src_image": raw["source_image"],
    "trg_image": raw["target_image"],
    "src_x": raw["source_x"],
    "src_y": raw["source_y"],
})

adapted_path = Path("/kaggle/working/test_adapted.csv")
adapted.to_csv(adapted_path, index=False)

In [12]:
from soco.submit import predict_split

final = ft_cfg

test_cfg = final.replace(
    data_format="csv",
    max_pairs=0,
    csv_path=str(adapted_path),
    image_dir=str(root),
)

submission = predict_split(
    test_cfg,
    split="test",
    out_csv="/kaggle/working/submission.csv",
    sample_submission=str(root / "sample_submission.csv"),
)

assert len(submission) == len(raw)
assert submission["id"].is_unique

predict:   0%|          | 0/577 [00:00<?, ?it/s]

feature cache: 1025 hits / 129 misses
wrote 577 rows -> /kaggle/working/submission.csv


In [13]:
ref = pd.read_csv(root / "sample_submission.csv")

assert submission.columns.tolist() == ["id", "prediction"]
assert submission["id"].tolist() == ref["id"].tolist()
assert submission["id"].is_unique

xy = np.asarray(
    [json.loads(value) for value in submission["prediction"]],
    dtype=np.float64,
)

assert xy.shape == (len(ref), 2)
assert np.isfinite(xy).all()